# Pipeline completo en Colab
Ingesta → perfiles → modelo de riesgo → cartografía → agente de intervención → app. Descarga los datos públicos del INE (~3.5 GB) y la cartografía. Ejecuta las celdas en orden.

In [1]:
!git clone https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd ParticipacionCiudadanaINE
!pip -q install -r requirements.txt

Cloning into 'ParticipacionCiudadanaINE'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 43 (delta 10), reused 42 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 2.33 MiB | 23.15 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/ParticipacionCiudadanaINE


In [2]:
%%bash
set -e
mkdir -p data/raw/zips && cd data/raw/zips
for p in 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2009.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2012.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2015.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2018.zip 2022/04/DatosAbiertos_DECEyEC_ConteosCensales2021.zip 2025/08/DatosAbiertos_DECEyEC_ConteosCensales2024.zip; do
  curl -sL -A 'Mozilla/5.0' -O "https://ine.mx/wp-content/uploads/$p"
done
for z in *.zip; do y=$(echo $z | grep -oE '[0-9]{4}'); mkdir -p ../$y && unzip -oq "$z" -d ../$y; done
rm -f *.zip
cd ../../.. && mkdir -p data/geo && cd data/geo
for f in SECCION MUNICIPIO ENTIDAD; do curl -sL -A 'Mozilla/5.0' -O "https://pautas.ine.mx/transparencia/mapas/cob_2025/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f && rm $f.zip; done
ls ../raw ../geo

../geo:
ENTIDAD
MUNICIPIO
SECCION

../raw:
2009
2012
2015
2018
2021
2024
zips


In [3]:
!python -m src.ingest

{
  "filas": 406866,
  "secciones": 72681,
  "secciones_en_todas": 62048,
  "ns_max": 0.5,
  "por_eleccion": {
    "2009": {
      "secciones": 64934,
      "validas": 63546,
      "pct_validas": 0.9786,
      "participacion_nacional": 0.4406,
      "participacion_mediana_seccion": 0.443
    },
    "2012": {
      "secciones": 65599,
      "validas": 65484,
      "pct_validas": 0.9982,
      "participacion_nacional": 0.6208,
      "participacion_mediana_seccion": 0.6328
    },
    "2015": {
      "secciones": 68362,
      "validas": 65328,
      "pct_validas": 0.9556,
      "participacion_nacional": 0.4707,
      "participacion_mediana_seccion": 0.4736
    },
    "2018": {
      "secciones": 68408,
      "validas": 64865,
      "pct_validas": 0.9482,
      "participacion_nacional": 0.6237,
      "participacion_mediana_seccion": 0.6411
    },
    "2021": {
      "secciones": 68812,
      "validas": 66455,
      "pct_validas": 0.9657,
      "participacion_nacional": 0.5184,
      "partic

In [4]:
!python -W ignore -m src.profiles --objetivo 2024

Ventana [2009, 2012, 2015, 2018, 2021] | secciones elegibles 67,738 de 69,906
   sil_kmeans  sil_gmm  sil_ward  db_kmeans  ch_kmeans      bic_gmm  ari_kmeans_gmm  ari_kmeans_ward  ari_kmeans_semilla
k                                                                                                                       
3       0.184    0.068     0.141      1.679   3306.747  1176767.558           0.104            0.453               0.996
4       0.168    0.055     0.093      1.626   3032.562  1156814.231           0.147            0.338               0.961
5       0.171    0.020     0.095      1.535   2790.150  1138107.026           0.140            0.343               0.983
6       0.165    0.014     0.086      1.559   2614.745  1127742.894           0.215            0.372               0.982
7       0.153    0.005     0.091      1.565   2442.234  1116703.769           0.193            0.306               0.990
8       0.154   -0.010     0.093      1.540   2282.069  1102619.626        

In [ ]:
!python -W ignore -m src.risk_model

Filas por objetivo: {2015: 66679, 2018: 68310, 2021: 68199, 2024: 69193, 2027: 71208} (11s)
  [valid 2021] lineal {'alpha': 0.1} -> MAE 6.852 MAE_rel 6.408
  [valid 2021] lineal {'alpha': 1.0} -> MAE 6.852 MAE_rel 6.408
  [valid 2021] lineal {'alpha': 10.0} -> MAE 6.849 MAE_rel 6.409
  [valid 2021] lineal {'alpha': 100.0} -> MAE 6.831 MAE_rel 6.412
  [valid 2021] random_forest {'min_samples_leaf': 10, 'max_features': 0.33} -> MAE 7.159 MAE_rel 7.1
  [valid 2021] random_forest {'min_samples_leaf': 10, 'max_features': 0.6} -> MAE 7.024 MAE_rel 7.243
  [valid 2021] random_forest {'min_samples_leaf': 50, 'max_features': 0.33} -> MAE 7.249 MAE_rel 7.191
  [valid 2021] random_forest {'min_samples_leaf': 50, 'max_features': 0.6} -> MAE 7.106 MAE_rel 7.311
  [valid 2021] gradient_boosting {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 50} -> MAE 7.089 MAE_rel 7.345
  [valid 2021] gradient_boosting {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 200} -> MAE 

## Fase 4: cartografía ligera, agente y app

In [ ]:
!python -W ignore -m src.geo

In [ ]:
!python -m src.agent --anio 2027 --presupuesto-pct 10

### App (Streamlit)
Abre la URL `https://….trycloudflare.com` que se imprime al final. Para detenerla: *Entorno de ejecución → Interrumpir*.

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
sleep 12
grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1

In [ ]:
!cat data/processed/riesgo_metricas.json | head -c 6000

In [ ]:
!zip -qr processed.zip data/processed
from google.colab import files
files.download('processed.zip')